In [6]:
LAST_NAME = "Wales"
STUDENT_ID = "TUPM-26-7925"
FAVORITE_ARTIST = "IVOS"

id_digits = [int(c) for c in STUDENT_ID if c.isdigit()]
letters = [c for c in FAVORITE_ARTIST.upper() if c.isalpha()]

SEED_NUM = id_digits[-1]    # CHECK this against the SEED_NUM rule in your Lab 4 handout

print("SEED_NUM        :", SEED_NUM)
print("LAST_NAME       :", LAST_NAME)
print("FAVORITE_ARTIST :", FAVORITE_ARTIST)

SEED_NUM        : 5
LAST_NAME       : Wales
FAVORITE_ARTIST : IVOS


In [7]:
sources = [f"{LAST_NAME}-T{i}" for i in range(1, 6)]
categories = ["TEMPERATURE", "VOLTAGE", "CURRENT", "PRESSURE", "SPEED"]

counter = 0
def next_value():
    """Value from SEED_NUM and the next character position of FAVORITE_ARTIST."""
    global counter
    pos = counter % len(letters) + 1                    # character position (1-based)
    letter_value = ord(letters[pos - 1]) - ord("A") + 1  # A=1 ... Z=26
    counter += 1
    return letter_value * pos + SEED_NUM

records = []

# one record per source, each with a different category
for i in range(5):
    records.append({"source": sources[i], "category": categories[i], "value": next_value()})

# extra measurements: TEMPERATURE, VOLTAGE and CURRENT get multiple readings
for s, cat in [(0, "TEMPERATURE"), (1, "TEMPERATURE"), (1, "VOLTAGE"), (2, "VOLTAGE"), (3, "CURRENT")]:
    records.append({"source": sources[s], "category": cat, "value": next_value()})

# one repeated record (exact copy of the first record)
records.append(dict(records[0]))

print("Sources   :", sources)
print("Categories:", categories)
print()
print(f"{'#':<4}{'Source':<12}{'Category':<14}{'Value':>6}")
for n, r in enumerate(records, 1):
    print(f"{n:<4}{r['source']:<12}{r['category']:<14}{r['value']:>6}")

Sources   : ['Wales-T1', 'Wales-T2', 'Wales-T3', 'Wales-T4', 'Wales-T5']
Categories: ['TEMPERATURE', 'VOLTAGE', 'CURRENT', 'PRESSURE', 'SPEED']

#   Source      Category       Value
1   Wales-T1    TEMPERATURE       14
2   Wales-T2    VOLTAGE           49
3   Wales-T3    CURRENT           50
4   Wales-T4    PRESSURE          81
5   Wales-T5    SPEED             14
6   Wales-T1    TEMPERATURE       49
7   Wales-T2    TEMPERATURE       50
8   Wales-T2    VOLTAGE           81
9   Wales-T3    VOLTAGE           14
10  Wales-T4    CURRENT           49
11  Wales-T1    TEMPERATURE       14


In [8]:
organized = {}
for r in records:
    organized.setdefault(r["source"], {}).setdefault(r["category"], []).append(r["value"])

for src, cats in organized.items():
    print(src)
    for cat, vals in cats.items():
        print(f"   {cat:<12}: {vals}")

Wales-T1
   TEMPERATURE : [14, 49, 14]
Wales-T2
   VOLTAGE     : [49, 81]
   TEMPERATURE : [50]
Wales-T3
   CURRENT     : [50]
   VOLTAGE     : [14]
Wales-T4
   PRESSURE    : [81]
   CURRENT     : [49]
Wales-T5
   SPEED       : [14]


In [9]:
from collections import Counter

counts = Counter((r["source"], r["category"], r["value"]) for r in records)
repeated = [k for k, c in counts.items() if c > 1]
distinct = set(counts)

print("Total records   :", len(records))
print("Distinct records:", len(distinct))
print("Repeated records:", repeated)
print()
print("Distinct sources   :", sorted({r["source"] for r in records}))
print("Distinct categories:", sorted({r["category"] for r in records}))
print("Distinct values    :", sorted({r["value"] for r in records}))

Total records   : 11
Distinct records: 10
Repeated records: [('Wales-T1', 'TEMPERATURE', 14)]

Distinct sources   : ['Wales-T1', 'Wales-T2', 'Wales-T3', 'Wales-T4', 'Wales-T5']
Distinct categories: ['CURRENT', 'PRESSURE', 'SPEED', 'TEMPERATURE', 'VOLTAGE']
Distinct values    : [14, 49, 50, 81]


In [10]:
by_category = {}
for r in records:
    by_category.setdefault(r["category"], []).append(r["value"])

cat_summary = {}
print(f"{'Category':<14}{'Count':>6}{'Min':>6}{'Max':>6}{'Sum':>6}{'Avg':>8}")
for cat, vals in by_category.items():
    cat_summary[cat] = {"count": len(vals), "min": min(vals), "max": max(vals),
                        "sum": sum(vals), "avg": sum(vals) / len(vals)}
    s = cat_summary[cat]
    print(f"{cat:<14}{s['count']:>6}{s['min']:>6}{s['max']:>6}{s['sum']:>6}{s['avg']:>8.2f}")

print()
print("Records per source:", dict(Counter(r["source"] for r in records)))
all_values = [r["value"] for r in records]
print("Overall min / max / avg:", min(all_values), max(all_values), round(sum(all_values) / len(all_values), 2))

Category       Count   Min   Max   Sum     Avg
TEMPERATURE        4    14    50   127   31.75
VOLTAGE            3    14    81   144   48.00
CURRENT            2    49    50    99   49.50
PRESSURE           1    81    81    81   81.00
SPEED              1    14    14    14   14.00

Records per source: {'Wales-T1': 3, 'Wales-T2': 3, 'Wales-T3': 2, 'Wales-T4': 2, 'Wales-T5': 1}
Overall min / max / avg: 14 81 42.27


In [11]:
processed = {(src, cat): sum(v) / len(v)
             for src, cats in organized.items() for cat, v in cats.items()}

print("Processed results (average per source and category):")
for (src, cat), avg in processed.items():
    print(f"  {src:<12}{cat:<14}{avg:>8.2f}")

print("\nOrdered results (highest average first):")
for rank, ((src, cat), avg) in enumerate(sorted(processed.items(), key=lambda x: x[1], reverse=True), 1):
    print(f"  {rank}. {src:<12}{cat:<14}{avg:>8.2f}")

print("\nRecords ordered by source, category, value:")
for r in sorted(records, key=lambda r: (r["source"], r["category"], r["value"])):
    print(f"  {r['source']:<12}{r['category']:<14}{r['value']:>6}")

Processed results (average per source and category):
  Wales-T1    TEMPERATURE      25.67
  Wales-T2    VOLTAGE          65.00
  Wales-T2    TEMPERATURE      50.00
  Wales-T3    CURRENT          50.00
  Wales-T3    VOLTAGE          14.00
  Wales-T4    PRESSURE         81.00
  Wales-T4    CURRENT          49.00
  Wales-T5    SPEED            14.00

Ordered results (highest average first):
  1. Wales-T4    PRESSURE         81.00
  2. Wales-T2    VOLTAGE          65.00
  3. Wales-T2    TEMPERATURE      50.00
  4. Wales-T3    CURRENT          50.00
  5. Wales-T4    CURRENT          49.00
  6. Wales-T1    TEMPERATURE      25.67
  7. Wales-T3    VOLTAGE          14.00
  8. Wales-T5    SPEED            14.00

Records ordered by source, category, value:
  Wales-T1    TEMPERATURE       14
  Wales-T1    TEMPERATURE       14
  Wales-T1    TEMPERATURE       49
  Wales-T2    TEMPERATURE       50
  Wales-T2    VOLTAGE           49
  Wales-T2    VOLTAGE           81
  Wales-T3    CURRENT           50

In [13]:
highest = max(cat_summary, key=lambda c: cat_summary[c]["avg"])
lowest = min(cat_summary, key=lambda c: cat_summary[c]["avg"])

print("=== FINAL ORGANIZED AND SUMMARIZED TEST DATASET ===")
print("Total records     :", len(records))
print("Distinct records  :", len(distinct))
print("Repeated records  :", len(records) - len(distinct))
print("Highest avg category:", highest, "->", round(cat_summary[highest]["avg"], 2))
print("Lowest avg category :", lowest, "->", round(cat_summary[lowest]["avg"], 2))

=== FINAL ORGANIZED AND SUMMARIZED TEST DATASET ===
Total records     : 11
Distinct records  : 10
Repeated records  : 1
Highest avg category: PRESSURE -> 81.0
Lowest avg category : SPEED -> 14.0
